# StructflyDocTruth Flow Testing

This notebook configures DSPy with the current `.env` settings, loads sample files from `notebooks/data`, and runs the workflow in both text mode and `dspy.File` mode.

## Supported sample artifacts

- `invoice_summary.txt`: plain text invoice summary for the text-first flow
- `invoice_summary.pdf`: minimal PDF sample
- `invoice_summary.csv`: CSV spreadsheet sample
- `invoice_summary.xlsx`: minimal XLSX spreadsheet sample
- `invoice_summary.xls`: Excel-compatible XML spreadsheet sample
- `invoice_summary.docx`: minimal DOCX sample
- `invoice_summary.msg`: message-style sample for email testing

The `.msg` sample is a lightweight message payload for exercising the pipeline. Replace it with a real Outlook `.msg` file when you want production-like testing.

In [ ]:
from pathlib import Path
import json
import sys
from pprint import pprint

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = NOTEBOOK_DIR / "data"
sorted(path.name for path in DATA_DIR.iterdir() if path.is_file())

## Configure DSPy

This uses the project defaults from `.env`. At this point the configured model should be `ollama_chat/qwen2.5vl:7b` unless you changed it.

In [ ]:
from app.core.dspy_config import configure_dspy
from app.core.settings import get_settings
from app.core.file_utils import infer_file_metadata
from app.graph.workflow import graph
import dspy

settings = get_settings()
lm = configure_dspy(settings)

print({
    "dspy_model": settings.dspy_model_identifier,
    "dspy_api_base": settings.dspy_api_base,
    "default_source_type": settings.default_source_type,
})

## Text Flow Example

Use the `.txt` sample when you want to test deterministic key-value extraction plus DSPy field proposal behavior on already-readable text.

In [ ]:
def load_text_document(name: str) -> str:
    return (DATA_DIR / name).read_text(encoding="utf-8")

text_sample_name = "invoice_summary.txt"
text_sample = load_text_document(text_sample_name)
print(text_sample)

In [ ]:
def run_text_flow(document_name: str, source_type: str = "text") -> dict:
    text = load_text_document(document_name)
    return graph.invoke({"raw_text": text, "source_type": source_type})

text_result = run_text_flow(text_sample_name)
pprint(text_result["ground_truth_record"])

## File Flow Example

Use this path for `.pdf`, `.txt`, `.docx`, and `.msg`. The notebook mirrors the API behavior by creating a `dspy.File` directly from the file bytes and inferred MIME type.

In [ ]:
def build_dspy_file(path: Path):
    mime_type, source_type = infer_file_metadata(path.name)
    return (
        dspy.File.from_bytes(
            file_bytes=path.read_bytes(),
            filename=path.name,
            mime_type=mime_type,
        ),
        mime_type,
        source_type,
    )

def run_file_flow(document_name: str) -> dict:
    path = DATA_DIR / document_name
    document_file, mime_type, source_type = build_dspy_file(path)
    return graph.invoke(
        {
            "document_file": document_file,
            "filename": path.name,
            "mime_type": mime_type,
            "source_type": source_type,
        }
    )

file_sample_name = "invoice_summary.pdf"
file_result = run_file_flow(file_sample_name)
pprint(file_result["ground_truth_record"])

## Batch Run Across All Supported Samples

In [ ]:
batch_results = {}

for sample_path in sorted(DATA_DIR.glob("invoice_summary.*")):
    if sample_path.suffix == ".txt":
        batch_results[sample_path.name] = run_text_flow(sample_path.name)
    else:
        batch_results[sample_path.name] = run_file_flow(sample_path.name)

print(json.dumps(batch_results, indent=2, default=str))